<a href="https://www.kaggle.com/code/ferntanyl/big-data?scriptVersionId=358158021" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Replikasi Parsial: Deteksi Aritmia EKG dengan Apache Spark Structured Streaming

**Mata kuliah:** Mahadata (IF25-40402)  
**Jenis eksperimen:** replikasi parsial dengan rekaman EKG dan anotasi MIT-BIH  
**Platform:** Kaggle Notebook, PySpark mode lokal

## Artikel acuan

Ilbeigipour, S., Albadvi, A., & Akhondzadeh Noughabi, E. (2021). *Real-Time Heart Arrhythmia Detection Using Apache Spark Structured Streaming*. **Journal of Healthcare Engineering, 2021**, Article ID 6624829. https://doi.org/10.1155/2021/6624829

Artikel tersebut melakukan preprocessing sinyal EKG, segmentasi beat, ekstraksi fitur DWT, membandingkan Decision Tree, Logistic Regression, dan Random Forest, lalu menggunakan Random Forest dalam Spark Structured Streaming.

## Batas eksperimen

Notebook ini memakai dataset **`mondejar/mitbih-database`**, yaitu rekaman EKG CSV dan anotasi beat TXT. Ia membentuk potongan sinyal di sekitar posisi beat berdasarkan anotasi, lalu membandingkan tiga model Spark MLlib dan mensimulasikan pemrosesan file micro-batch.

Ini adalah replikasi parsial, bukan replikasi persis: fitur yang digunakan di sini adalah sampel sinyal pada jendela beat, bukan 25 fitur hasil DWT seperti artikel. Kelas juga dipetakan ke lima kelompok AAMI bila anotasi tersedia. Hasil karena itu tidak boleh diklaim identik atau dibandingkan langsung dengan metrik artikel tanpa menjelaskan perbedaannya.

Eksperimen: (1) membaca rekaman dan anotasi, (2) membentuk potongan beat, (3) memisahkan data berdasarkan kelompok rekaman, (4) melatih Decision Tree, Logistic Regression, dan Random Forest, (5) mencatat metrik dan waktu, serta (6) mensimulasikan Structured Streaming pada satu mesin.


## 1. Persiapan lingkungan

In [8]:
import os, sys, glob, time, json, platform, subprocess, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

SEED = 42
OUT_DIR = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "./results"
os.makedirs(OUT_DIR, exist_ok=True)

spark = (SparkSession.builder
         .appName("MITBIH-Arrhythmia-Partial-Replication")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Python:", platform.python_version())
print("PySpark:", __import__("pyspark").__version__)
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Output:", OUT_DIR)


Python: 3.13.15
PySpark: 4.0.4
Spark: 4.0.4
Master: local[*]
Output: /kaggle/working/results


## 2. Temukan dataset Mondejar di Kaggle

Pastikan Anda sudah menambahkan **`mondejar/mitbih-database`** melalui tombol **Add Data**. Kode mencari `100.csv` secara rekursif sehingga tidak bergantung pada path persis. Ia juga memeriksa apakah berkas anotasi pasangannya tersedia.


In [9]:
# Dataset yang dipakai: mondejar/mitbih-database
import re

# Daftar rekaman standar untuk pemisahan per rekaman (agar beat dari rekaman yang sama
# tidak bocor sekaligus ke data train dan test).
DS1 = set("101 106 108 109 112 114 115 116 118 119 122 124 201 203 205 207 208 209 215 220 223 230".split())
DS2 = set("100 103 105 111 113 117 121 123 200 202 210 212 213 214 219 221 222 228 231 232 233 234".split())
PRE, POST = 90, 110  # 200 sampel per beat; MIT-BIH umumnya disampel pada 360 Hz
AAMI = {
    "N":"N", "L":"N", "R":"N", "e":"N", "j":"N",
    "A":"S", "a":"S", "J":"S", "S":"S",
    "V":"V", "E":"V", "F":"F", "/":"Q", "f":"Q", "Q":"Q"
}
BEAT_SYMBOLS = set(AAMI)

all_csv = sorted(glob.glob("/kaggle/input/**/100.csv", recursive=True))
if not all_csv:
    candidates = glob.glob("/kaggle/input/**/*", recursive=True)
    files = [p for p in candidates if os.path.isfile(p)]
    print("Contoh file yang ditemukan di /kaggle/input:")
    print("\n".join(files[:60]) if files else "Tidak ada file input terdeteksi.")
    raise FileNotFoundError(
        "100.csv tidak ditemukan. Pastikan dataset mondejar/mitbih-database sudah ditambahkan "
        "melalui Add Data dan status notebook bukan tanpa dataset input."
    )
DATA_DIR = os.path.dirname(all_csv[0])

sig_files = sorted(glob.glob(os.path.join(DATA_DIR, "[0-9][0-9][0-9].csv")))
ann_files = sorted(glob.glob(os.path.join(DATA_DIR, "[0-9][0-9][0-9]annotations.txt")))
# Jika file berada pada subfolder yang berbeda, cari ulang secara rekursif dari input.
if not sig_files:
    sig_files = sorted(glob.glob("/kaggle/input/**/[0-9][0-9][0-9].csv", recursive=True))
if not ann_files:
    ann_files = sorted(glob.glob("/kaggle/input/**/*annotations.txt", recursive=True))

sig_map = {os.path.basename(p)[:3]: p for p in sig_files}
ann_map = {os.path.basename(p)[:3]: p for p in ann_files}
common_records = sorted(set(sig_map) & set(ann_map))
print("Direktori sinyal yang terdeteksi:", DATA_DIR)
print("Jumlah file sinyal:", len(sig_files), "| file anotasi:", len(ann_files))
print("Rekaman dengan pasangan sinyal + anotasi:", len(common_records))
print("Contoh rekaman:", common_records[:15])
if not common_records:
    raise FileNotFoundError(
        "File CSV dan annotations.txt tidak ditemukan sebagai pasangan. "
        "Periksa struktur dataset dengan melihat daftar file di /kaggle/input."
    )

# Gunakan split rekaman standar bila tersedia. Jika dataset hanya berisi subset, gunakan
# split rekaman deterministik alternatif (bukan split acak per beat).
train_records = sorted(set(common_records) & DS1)
test_records = sorted(set(common_records) & DS2)
if not train_records or not test_records:
    fallback = sorted(common_records)
    cut = max(1, int(len(fallback) * 0.7))
    train_records, test_records = fallback[:cut], fallback[cut:]
    if not test_records:
        raise ValueError("Dataset harus memiliki setidaknya dua rekaman beranotasi untuk train/test.")
    print("PERINGATAN: rekaman DS1/DS2 tidak lengkap; memakai split berdasarkan ID rekaman yang tersedia.")
print("Rekaman train:", train_records)
print("Rekaman test :", test_records)

# Tampilkan contoh format untuk membantu diagnosis jika file punya skema berbeda.
with open(sig_map[common_records[0]], encoding="utf-8", errors="replace") as f:
    print("\nContoh baris CSV:")
    for _ in range(3):
        line = f.readline()
        if not line: break
        print(line.rstrip())
with open(ann_map[common_records[0]], encoding="utf-8", errors="replace") as f:
    print("\nContoh baris anotasi:")
    for _ in range(6):
        line = f.readline()
        if not line: break
        print(line.rstrip())

# Parse annotation lines of the common form: time  sample  symbol  ...
ANN_RE = re.compile(r"^\s*([\d:.]+)\s+(\d+)\s+(\S+)")

def read_signal_csv(path):
    frame = pd.read_csv(path)
    # Deteksi CSV tanpa header: bila nama kolomnya sendiri tampak seperti angka,
    # baca ulang tanpa header agar baris pertama tidak hilang.
    if all(re.fullmatch(r"[-+]?\d+(?:\.\d+)?", str(c).strip()) for c in frame.columns):
        frame = pd.read_csv(path, header=None)
    frame = frame.apply(pd.to_numeric, errors="coerce")
    frame = frame.dropna(axis=1, how="all").dropna(axis=0, how="all")
    if frame.shape[1] < 2:
        raise ValueError(f"Format CSV tidak memiliki minimal dua kolom numerik: {path}")
    # Kolom pertama adalah nomor sampel; sinyal menggunakan kolom setelahnya.
    # Pilih kanal pertama yang tersedia agar format 2- maupun 3-kolom dapat dibaca.
    sample_values = frame.iloc[:, 0].to_numpy()
    signal_values = frame.iloc[:, 1].to_numpy(dtype=float)
    if frame.shape[1] >= 3:
        # Pilih kanal kedua jika kanal pertama konstan/tidak layak; selain itu gunakan kanal pertama.
        candidate = frame.iloc[:, 2].to_numpy(dtype=float)
        if np.nanstd(signal_values) == 0 and np.nanstd(candidate) > 0:
            signal_values = candidate
    # Umumnya kolom pertama berisi index sampel 0..N-1. Jika tidak, anotasi MIT-BIH
    # tetap ditafsirkan sebagai posisi sampel (offset dari awal rekaman).
    return signal_values

def read_annotations(path):
    annotations = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = ANN_RE.match(line)
            if m:
                sample_idx = int(m.group(2))
                symbol = m.group(3)
                if symbol in BEAT_SYMBOLS:
                    annotations.append((sample_idx, symbol))
    return annotations

def extract_beats(record_id, signal_path, annotation_path):
    signal = read_signal_csv(signal_path)
    ann = read_annotations(annotation_path)
    rows = []
    for sample_idx, symbol in ann:
        start, end = sample_idx - PRE, sample_idx + POST
        if start < 0 or end > len(signal):
            continue
        segment = signal[start:end]
        if len(segment) != PRE + POST or not np.isfinite(segment).all():
            continue
        label = AAMI[symbol]
        rows.append((record_id, label, *segment.astype(float).tolist()))
    return rows

rows_train, rows_test = [], []
for rid in train_records:
    rows_train.extend(extract_beats(rid, sig_map[rid], ann_map[rid]))
for rid in test_records:
    rows_test.extend(extract_beats(rid, sig_map[rid], ann_map[rid]))

if not rows_train or not rows_test:
    raise ValueError(
        "Segmentasi beat menghasilkan 0 baris pada train atau test. Kemungkinan format CSV/anotasi "
        "berbeda dari asumsi kode; periksa contoh baris yang dicetak pada sel ini."
    )
feature_cols = [f"x{i:03d}" for i in range(PRE + POST)]
n_features = len(feature_cols)
all_cols = ["record", "label_name"] + feature_cols
train_pd = pd.DataFrame(rows_train, columns=all_cols)
test_pd = pd.DataFrame(rows_test, columns=all_cols)
label_to_int = {name: i for i, name in enumerate(["N", "S", "V", "F", "Q"])}
train_pd["label"] = train_pd["label_name"].map(label_to_int).astype(int)
test_pd["label"] = test_pd["label_name"].map(label_to_int).astype(int)
train_pd = train_pd.drop(columns=["label_name"])
test_pd = test_pd.drop(columns=["label_name"])
print("Jumlah beat train:", len(train_pd), "| test:", len(test_pd))
print("Jumlah fitur sampel per beat:", len(feature_cols))
print("Distribusi label train:\n", train_pd["label"].value_counts().sort_index())
print("Distribusi label test:\n", test_pd["label"].value_counts().sort_index())
display(train_pd.head())


Direktori sinyal yang terdeteksi: /kaggle/input/datasets/mondejar/mitbih-database
Jumlah file sinyal: 48 | file anotasi: 48
Rekaman dengan pasangan sinyal + anotasi: 48
Contoh rekaman: ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115']
Rekaman train: ['101', '106', '108', '109', '112', '114', '115', '116', '118', '119', '122', '124', '201', '203', '205', '207', '208', '209', '215', '220', '223', '230']
Rekaman test : ['100', '103', '105', '111', '113', '117', '121', '123', '200', '202', '210', '212', '213', '214', '219', '221', '222', '228', '231', '232', '233', '234']

Contoh baris CSV:
'sample #','MLII','V5'
0,995,1011
1,995,1011

Contoh baris anotasi:
      Time   Sample #  Type  Sub Chan  Num	Aux
    0:00.050       18     +    0    0    0	(N
    0:00.214       77     N    0    0    0
    0:01.028      370     N    0    0    0
    0:01.839      662     N    0    0    0
    0:02.628      946     N    0    0    0
Jumlah beat train

,record,x000,x001,x002,x003,x004,x005,x006,x007,x008,...,x191,x192,x193,x194,x195,x196,x197,x198,x199,label
0,101,954.0,958.0,957.0,959.0,962.0,956.0,953.0,957.0,960.0,...,1001.0,998.0,1000.0,1000.0,999.0,1001.0,998.0,999.0,999.0,0
1,101,971.0,969.0,970.0,969.0,969.0,969.0,972.0,971.0,969.0,...,997.0,1002.0,998.0,999.0,998.0,997.0,1000.0,1000.0,1000.0,0
2,101,947.0,948.0,952.0,955.0,955.0,951.0,949.0,947.0,953.0,...,975.0,976.0,982.0,985.0,988.0,986.0,985.0,986.0,987.0,0
3,101,940.0,942.0,944.0,943.0,940.0,940.0,938.0,941.0,944.0,...,978.0,977.0,979.0,983.0,985.0,985.0,985.0,984.0,988.0,0
4,101,962.0,964.0,960.0,960.0,958.0,959.0,962.0,960.0,961.0,...,953.0,960.0,963.0,960.0,959.0,958.0,961.0,965.0,964.0,0


## 3. Muat beat hasil segmentasi ke Spark dan periksa distribusi kelas

Setiap baris pada DataFrame merupakan satu beat yang dipotong dari sinyal mentah menggunakan indeks anotasi. Kolom `label` adalah label numerik hasil pemetaan simbol anotasi ke kelompok AAMI: N, S, V, F, dan Q.


In [10]:
train = spark.createDataFrame(train_pd).cache()
test = spark.createDataFrame(test_pd).cache()

train_count, test_count = train.count(), test.count()
print(f"Jumlah beat train: {train_count:,}")
print(f"Jumlah beat test : {test_count:,}")

label_names = {0: "N - normal", 1: "S - supraventricular ectopic", 2: "V - ventricular ectopic", 3: "F - fusion", 4: "Q - unknown/other"}

def class_distribution(df, split_name):
    return (df.groupBy("label").count().orderBy("label").toPandas()
            .assign(split=split_name, class_name=lambda d: d["label"].map(label_names).fillna("label lain")))

dist = pd.concat([class_distribution(train, "train"), class_distribution(test, "test")], ignore_index=True)
display(dist)
dist.to_csv(os.path.join(OUT_DIR, "01_distribusi_kelas.csv"), index=False)
print("Jumlah nilai null pada fitur train:", int(train.select([F.sum(F.col(c).isNull().cast("long")).alias(c) for c in feature_cols]).toPandas().sum(axis=1).iloc[0]))


Jumlah beat train: 51,008
Jumlah beat test : 49,697


,label,count,split,class_name
0,0,45854,train,N - normal
1,1,944,train,S - supraventricular ectopic
2,2,3788,train,V - ventricular ectopic
3,3,414,train,F - fusion
4,4,8,train,Q - unknown/other
5,0,44245,test,N - normal
6,1,1837,test,S - supraventricular ectopic
7,2,3220,test,V - ventricular ectopic
8,3,388,test,F - fusion
9,4,7,test,Q - unknown/other


Jumlah nilai null pada fitur train: 0


## 4. Jalankan tiga model seperti artikel

In [11]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

models = {
    "Decision Tree": DecisionTreeClassifier(
        labelCol="label", featuresCol="features", maxDepth=8, seed=SEED
    ),
    "Logistic Regression": LogisticRegression(
        labelCol="label", featuresCol="features", maxIter=60, regParam=0.01,
        family="multinomial"
    ),
    "Random Forest": RandomForestClassifier(
        labelCol="label", featuresCol="features", numTrees=40, maxDepth=10, seed=SEED
    )
}

# Evaluasi prediksi dengan macro-F1 dan accuracy dari sklearn jika tersedia.
try:
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
    HAVE_SKLEARN = True
except ImportError:
    HAVE_SKLEARN = False

results = []
fitted_models = {}
for name, classifier in models.items():
    pipeline = Pipeline(stages=[assembler, classifier])
    t0 = time.perf_counter()
    fitted = pipeline.fit(train)
    fit_seconds = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred = fitted.transform(test).select("label", "prediction").toPandas()
    predict_seconds = time.perf_counter() - t1

    y_true = pred["label"].astype(int).to_numpy()
    y_pred = pred["prediction"].astype(int).to_numpy()
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    )
    pred_spark = fitted.transform(test).select("label", "prediction")
    accuracy = evaluator.evaluate(pred_spark)

    if HAVE_SKLEARN:
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
        macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    else:
        macro_f1 = float("nan")
        macro_precision = float("nan")
        macro_recall = float("nan")

    results.append({
        "model": name, "accuracy": accuracy, "macro_f1": macro_f1,
        "macro_precision": macro_precision, "macro_recall": macro_recall,
        "fit_seconds": fit_seconds, "predict_seconds": predict_seconds
    })
    fitted_models[name] = fitted
    print(f"{name}: accuracy={accuracy:.4f}, macro-F1={macro_f1:.4f}, "
          f"fit={fit_seconds:.1f}s, predict={predict_seconds:.1f}s")

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(results_df.round(4))
results_df.to_csv(os.path.join(OUT_DIR, "02_perbandingan_model.csv"), index=False)

best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
print("Model dengan macro-F1 tertinggi pada run ini:", best_name)


Decision Tree: accuracy=0.8776, macro-F1=0.3053, fit=13.4s, predict=2.2s


Logistic Regression: accuracy=0.8993, macro-F1=0.3067, fit=23.8s, predict=1.6s


Random Forest: accuracy=0.9070, macro-F1=0.3260, fit=16.4s, predict=2.0s


,model,accuracy,macro_f1,macro_precision,macro_recall,fit_seconds,predict_seconds
2,Random Forest,0.9070,0.3260,0.4157,0.3441,16.3961,1.9776
1,Logistic Regression,0.8993,0.3067,0.3875,0.2810,23.7912,1.5707
0,Decision Tree,0.8776,0.3053,0.2946,0.3218,13.4438,2.1835


Model dengan macro-F1 tertinggi pada run ini: Random Forest


## 5. Detail metrik model terbaik

In [12]:
best_pred = best_model.transform(test).select("label", "prediction").toPandas()
if HAVE_SKLEARN:
    print("Laporan klasifikasi model terbaik:", best_name)
    print(classification_report(
        best_pred["label"].astype(int),
        best_pred["prediction"].astype(int),
        labels=sorted(set(train_pd["label"].unique()) | set(test_pd["label"].unique())),
        target_names=[label_names.get(i, f"kelas {i}") for i in sorted(set(train_pd["label"].unique()) | set(test_pd["label"].unique()))],
        zero_division=0
    ))
else:
    print("scikit-learn tidak tersedia; macro precision/recall dan classification report dilewati.")


Laporan klasifikasi model terbaik: Random Forest
                              precision    recall  f1-score   support

                  N - normal       0.94      0.96      0.95     44245
S - supraventricular ectopic       0.52      0.01      0.01      1837
     V - ventricular ectopic       0.59      0.75      0.66      3220
                  F - fusion       0.03      0.01      0.01       388
           Q - unknown/other       0.00      0.00      0.00         7

                    accuracy                           0.91     49697
                   macro avg       0.42      0.34      0.33     49697
                weighted avg       0.89      0.91      0.89     49697



## 6. Simulasi Spark Structured Streaming

Artikel menjalankan query setiap 5 detik pada sinyal mentah dan mengirimkan hasil ke output sink. Bagian ini hanya **simulasi micro-batch**: data uji yang sudah berbentuk beat-level ditulis menjadi berkas Parquet, lalu Spark membacanya sebagai aliran berkas. Ini menguji jalur streaming Spark, bukan real-time klinis atau latency end-to-end dari sinyal mentah.

In [13]:
stream_input = os.path.join(OUT_DIR, "stream_input")
checkpoint = os.path.join(OUT_DIR, "stream_checkpoint")
import shutil
for path in [stream_input, checkpoint]:
    shutil.rmtree(path, ignore_errors=True)

# Beberapa partisi menjadi beberapa berkas Parquet; maxFilesPerTrigger membacanya bertahap.
test.select("label", *feature_cols).repartition(4).write.mode("overwrite").parquet(stream_input)
stream_schema = test.select("label", *feature_cols).schema
stream_df = (spark.readStream
             .schema(stream_schema)
             .option("maxFilesPerTrigger", 1)
             .parquet(stream_input))

stream_results = []
def process_batch(batch_df, batch_id):
    t0 = time.perf_counter()
    n_rows = batch_df.count()
    # Model menjalankan inferensi pada micro-batch; jumlah prediksi dihitung untuk memastikan proses berjalan.
    n_predictions = best_model.transform(batch_df).select("prediction").count()
    elapsed = time.perf_counter() - t0
    stream_results.append({
        "batch_id": int(batch_id),
        "rows_in": int(n_rows),
        "predictions": int(n_predictions),
        "processing_seconds": round(elapsed, 4),
        "ms_per_row": round(1000 * elapsed / max(n_rows, 1), 6)
    })

query = (stream_df.writeStream
         .foreachBatch(process_batch)
         .option("checkpointLocation", checkpoint)
         .trigger(availableNow=True)
         .start())
query.awaitTermination()

stream_results_df = pd.DataFrame(stream_results).sort_values("batch_id")
display(stream_results_df)
stream_results_df.to_csv(os.path.join(OUT_DIR, "03_streaming_microbatch.csv"), index=False)

if len(stream_results_df):
    print("Total beat yang diproses:", int(stream_results_df["rows_in"].sum()))
    print("Total waktu micro-batch (detik):", round(stream_results_df["processing_seconds"].sum(), 3))
else:
    print("Tidak ada micro-batch yang diproses. Periksa input Parquet dan log Spark.")


,batch_id,rows_in,predictions,processing_seconds,ms_per_row
0,0,12425,12425,1.7679,0.142283
1,1,12424,12424,0.9784,0.078750
2,2,12424,12424,0.7992,0.064327
3,3,12424,12424,0.8884,0.071508


Total beat yang diproses: 49697
Total waktu micro-batch (detik): 4.434


## 7. Simpan ringkasan eksperimen

In [14]:
summary = {
    "article": "Ilbeigipour et al. (2021), DOI:10.1155/2021/6624829",
    "replication_type": "partial replication using Mondejar MIT-BIH raw ECG recordings and beat annotations",
    "dataset": "mondejar/mitbih-database",
    "train_records": train_records,
    "test_records": test_records,
    "n_features_per_beat": int(n_features),
    "n_train_beats": int(train_count),
    "n_test_beats": int(test_count),
    "spark_version": spark.version,
    "pyspark_version": __import__("pyspark").__version__,
    "best_model_by_macro_f1": best_name,
    "model_results": results,
    "streaming_batches": stream_results,
    "limitations": [
        "Beat dipotong dari sinyal dengan jendela tetap 90 sampel sebelum dan 110 sesudah indeks anotasi.",
        "Fitur adalah sampel sinyal mentah, bukan 25 koefisien DWT seperti pada artikel.",
        "Label lima kelompok AAMI dan split rekaman yang tersedia tidak identik dengan konfigurasi artikel.",
        "Structured Streaming adalah simulasi file micro-batch pada satu mesin, bukan klaster multi-node atau sistem klinis real-time."
    ]
}
with open(os.path.join(OUT_DIR, "04_run_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Hasil tersimpan di:", OUT_DIR)
print("\n".join(sorted(os.listdir(OUT_DIR))))


Hasil tersimpan di: /kaggle/working/results
01_distribusi_kelas.csv
02_perbandingan_model.csv
03_streaming_microbatch.csv
04_run_summary.json
stream_checkpoint
stream_input


## 8. Catatan analisis untuk laporan

Isi bagian ini **setelah menjalankan semua sel**:

- **Lingkungan:** versi Python, PySpark, Spark, jumlah rekaman train/test, jumlah beat, dan jumlah fitur.
- **Prapemrosesan:** jelaskan jendela 90 sampel sebelum dan 110 sampel sesudah indeks anotasi serta pemetaan simbol menjadi lima label AAMI.
- **Hasil:** masukkan angka aktual dari `02_perbandingan_model.csv`; jangan mengganti hasil dengan angka artikel.
- **Streaming:** laporkan jumlah micro-batch, jumlah beat yang diproses, dan waktu pemrosesan aktual dari `03_streaming_microbatch.csv`. Ini simulasi file micro-batch, bukan latency yang identik dengan artikel.
- **Perbandingan dengan artikel:** artikel memakai preprocessing dan fitur DWT, sedangkan notebook ini memakai sampel sinyal mentah dalam jendela tetap. Karena fitur, label, split, dan lingkungan berbeda, angka tidak dapat dianggap perbandingan langsung.
- **Keterbatasan:** tidak melakukan FIR denoising atau DWT 25 fitur sebagaimana artikel; kualitas segmentasi bergantung pada format indeks anotasi; Spark berjalan lokal satu mesin.

**Referensi dataset:** Mondejar, *MIT-BIH Arrhythmia Database*, Kaggle dataset `mondejar/mitbih-database` (cantumkan URL dataset yang Anda gunakan di laporan/repo).

**Penting:** simpan notebook setelah *Run All* agar output aktual terlihat. Jika sel pencarian gagal, pastikan dataset sudah ditambahkan melalui Add Data. Jika parsing atau segmentasi gagal, periksa contoh baris CSV dan anotasi yang dicetak pada sel pencarian; jangan mengarang hasil eksperimen.
